# Learning Rate Schedulers & Cosine Annealing

A learning rate scheduler adjusts the learning rate during training according to a predefined rule.  
Choosing the right schedule is often as important as choosing the architecture.

## Why Does the Learning Rate Matter?

```
θ ← θ − α · ∇L(θ)
```

- **α too large** → overshoots the minimum, diverges
- **α too small** → slow convergence, gets stuck in flat regions
- **Scheduled α** → fast early progress + fine-grained convergence later

## Schedulers Covered

| # | Scheduler | Formula | Best For |
|---|---|---|---|
| 1 | StepLR | α × γ every N epochs | Simple baselines |
| 2 | ExponentialLR | α × γᵗ | Smooth decay |
| 3 | CosineAnnealingLR | ½(1 + cos(πt/T)) | Most modern training |
| 4 | CosineAnnealingWarmRestarts | SGDR with restarts | Ensemble-friendly |
| 5 | OneCycleLR | Warmup → peak → anneal | Super-convergence |
| 6 | LinearWarmup + Cosine | Custom transformer schedule | LLMs, ViTs |
| 7 | ReduceLROnPlateau | Halve when val loss stalls | Unknown dynamics |

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})

# ── Tiny model just to have an optimizer ──────────────────────────────
def make_opt(lr=0.1):
    model = nn.Linear(4, 1)
    return optim.SGD(model.parameters(), lr=lr)

def extract_lrs(scheduler, steps=100, plateau_losses=None):
    """Run scheduler for `steps` and collect LR at each step."""
    lrs = []
    for i in range(steps):
        lrs.append(scheduler.optimizer.param_groups[0]['lr'])
        if plateau_losses is not None:
            scheduler.step(plateau_losses[i])
        else:
            scheduler.step()
    return lrs

STEPS = 200
BASE_LR = 0.1
print("Setup complete.")

ModuleNotFoundError: No module named 'torch'

---
## 1. StepLR

Multiplies the learning rate by `gamma` every `step_size` epochs.

$$\alpha_t = \alpha_0 \cdot \gamma^{\lfloor t / \text{step\_size} \rfloor}$$

**Pros**: Simple, predictable  
**Cons**: Abrupt drops; requires manual tuning of step_size

In [ ]:
opt = make_opt(BASE_LR)
step_lrs = extract_lrs(
    optim.lr_scheduler.StepLR(opt, step_size=40, gamma=0.5),
    STEPS
)

plt.figure(figsize=(10, 3))
plt.plot(step_lrs, color='steelblue', linewidth=2)
plt.title('StepLR  (step_size=40, γ=0.5)')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.tight_layout(); plt.show()
print(f'LR drops at steps: {[i for i in range(1,STEPS) if step_lrs[i] < step_lrs[i-1]]}')

---
## 2. ExponentialLR

Decays the learning rate by `gamma` at every step.

$$\alpha_t = \alpha_0 \cdot \gamma^t$$

**Pros**: Smooth, continuous decay  
**Cons**: Can decay too fast for large `gamma` close to 1; LR reaches near-zero before training ends

In [ ]:
opt = make_opt(BASE_LR)
exp_lrs = extract_lrs(
    optim.lr_scheduler.ExponentialLR(opt, gamma=0.97),
    STEPS
)

plt.figure(figsize=(10, 3))
plt.plot(exp_lrs, color='darkorange', linewidth=2)
plt.title('ExponentialLR  (γ=0.97)')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.tight_layout(); plt.show()
print(f'Final LR: {exp_lrs[-1]:.6f}  (started at {exp_lrs[0]:.4f})')

---
## 3. CosineAnnealingLR — Deep Dive

The most widely used scheduler in modern deep learning (ResNets, ViTs, LLMs).

### Mathematics

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})\left(1 + \cos\left(\frac{\pi \cdot t}{T_{\max}}\right)\right)$$

where:
- $t$ = current step
- $T_{\max}$ = total steps in one cosine cycle
- $\alpha_{\min}$ = minimum learning rate (usually 0 or a small value)
- $\alpha_{\max}$ = initial (maximum) learning rate

### Why the Cosine Shape?

```
t=0:   cos(0)   = 1   →  α = αmax        (start fast)
t=T/2: cos(π/2) = 0   →  α = (αmax+αmin)/2  (halfway)
t=T:   cos(π)   = -1  →  α = αmin        (end slow)
```

The cosine curve has **slow change near the ends and fast change in the middle**,  
which means:
- **Early training**: α decreases gently → still exploring
- **Mid training**: α decreases quickly → converging fast
- **Late training**: α decreases slowly → fine-tuning near minimum

In [ ]:
# ── Manual cosine annealing to show the math transparently ───────────
def cosine_annealing_manual(alpha_max, alpha_min, T_max, steps):
    t = np.arange(steps)
    return alpha_min + 0.5 * (alpha_max - alpha_min) * (1 + np.cos(np.pi * t / T_max))

t = np.arange(STEPS)
manual = cosine_annealing_manual(alpha_max=BASE_LR, alpha_min=0.001, T_max=STEPS, steps=STEPS)

# ── PyTorch built-in ──────────────────────────────────────────────────
opt = make_opt(BASE_LR)
cosine_lrs = extract_lrs(
    optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS, eta_min=0.001),
    STEPS
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(manual, color='crimson', linewidth=2, label='Manual formula')
axes[0].plot(cosine_lrs, 'k--', linewidth=1.5, label='PyTorch CosineAnnealingLR')
axes[0].set_title('CosineAnnealingLR  (T_max=200, η_min=0.001)')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Learning Rate')
axes[0].legend()

# Show the underlying cosine curve
axes[1].plot(t, np.cos(np.pi * t / STEPS), color='teal', linewidth=2)
axes[1].set_title('Underlying cos(πt/T) curve')
axes[1].set_xlabel('t'); axes[1].set_ylabel('cos(πt/T)')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)

plt.tight_layout(); plt.show()

# Verify formulas match
max_diff = max(abs(m - p) for m, p in zip(manual, cosine_lrs))
print(f'Max difference (manual vs PyTorch): {max_diff:.2e}  ← numerically identical')

---
## 4. CosineAnnealingWarmRestarts (SGDR)

**Stochastic Gradient Descent with Warm Restarts** (Loshchilov & Hutter, 2017).

The cosine schedule **restarts periodically**. Each restart resets α to αmax.

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})\left(1 + \cos\left(\frac{\pi \cdot T_{\text{cur}}}{T_i}\right)\right)$$

where $T_i$ is the length of the i-th cycle (can grow with `T_mult`).

### Why Restarts Help
- The warm restarts help the model **escape sharp local minima**
- Each restart snapshot can be used for **model ensembling** at no extra training cost
- With `T_mult > 1`, cycles get progressively longer (more time near each minimum)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Fixed-length restarts every 50 steps
opt = make_opt(BASE_LR)
cosine_restart_lrs = extract_lrs(
    optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=1, eta_min=0.001),
    STEPS
)
axes[0].plot(cosine_restart_lrs, color='purple', linewidth=2)
axes[0].set_title('SGDR  (T_0=50, T_mult=1)  — fixed cycle length')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('LR')
for r in range(50, STEPS, 50):
    axes[0].axvline(r, color='gray', linestyle='--', alpha=0.5)

# Doubling cycle length (T_mult=2)
opt = make_opt(BASE_LR)
cosine_mult_lrs = extract_lrs(
    optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=25, T_mult=2, eta_min=0.001),
    STEPS
)
axes[1].plot(cosine_mult_lrs, color='darkgreen', linewidth=2)
axes[1].set_title('SGDR  (T_0=25, T_mult=2)  — doubling cycles')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('LR')

plt.tight_layout(); plt.show()

print('Snapshot ensemble checkpoints at warm-restart points.')
print('Each restart gives a model from a different basin → ensemble diversity.')

---
## 5. OneCycleLR

Smith & Touvron (2018) — **Super-Convergence**.

Three phases:
1. **Warmup** (0 → max_lr): Linear or cosine warmup
2. **Annealing** (max_lr → min_lr): Cosine annealing
3. **Final decay** (min_lr → min_lr/div_factor²): Very small LR for final polish

Key insight: cycling through a high LR phase **helps escape saddle points and sharp minima**,  
allowing convergence to wider, flatter minima that generalise better.

In [ ]:
opt = make_opt(BASE_LR)
one_cycle_lrs = extract_lrs(
    optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=BASE_LR,
        total_steps=STEPS,
        pct_start=0.3,          # 30% warmup
        anneal_strategy='cos',
        div_factor=25.0,        # initial LR = max_lr / 25
        final_div_factor=1e4,   # final LR = max_lr / (25 * 10000)
    ),
    STEPS
)

plt.figure(figsize=(10, 4))
plt.plot(one_cycle_lrs, color='darkorange', linewidth=2)
plt.axvline(int(0.3 * STEPS), color='gray', linestyle='--', alpha=0.7, label='Warmup end')
plt.title('OneCycleLR  (max_lr=0.1, pct_start=0.3)')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.legend(); plt.tight_layout(); plt.show()

warmup_end = int(0.3 * STEPS)
print(f'Initial LR  : {one_cycle_lrs[0]:.6f}  (max_lr / div_factor)')
print(f'Peak LR     : {max(one_cycle_lrs):.6f}  at step {one_cycle_lrs.index(max(one_cycle_lrs))}')
print(f'Final LR    : {one_cycle_lrs[-1]:.2e}')

---
## 6. Custom Linear Warmup + Cosine Annealing

This is the **de-facto transformer training schedule** used in BERT, GPT, T5, LLaMA.

$$
\alpha_t = \begin{cases}
\alpha_{\max} \cdot \dfrac{t}{T_{\text{warmup}}} & t < T_{\text{warmup}} \\[6pt]
\alpha_{\min} + \dfrac{1}{2}(\alpha_{\max} - \alpha_{\min})\left(1 + \cos\left(\dfrac{\pi(t - T_{\text{warmup}})}{T_{\max} - T_{\text{warmup}}}\right)\right) & t \geq T_{\text{warmup}}
\end{cases}
$$

**Warmup phase** prevents large gradient updates in early training when weights are random.  
**Cosine phase** smoothly anneals to near-zero.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

def get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps, min_lr_ratio=0.0):
    """
    Linear warmup for `warmup_steps`, then cosine annealing to
    `min_lr_ratio * base_lr` over the remaining steps.
    """
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            # Linear warmup: 0 → 1
            return float(current_step) / float(max(1, warmup_steps))
        # Cosine decay: 1 → min_lr_ratio
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine_decay = 0.5 * (1.0 + np.cos(np.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

    return LambdaLR(optimizer, lr_lambda)


WARMUP = 20  # first 20 steps = warmup

opt = make_opt(BASE_LR)
warmup_cosine_lrs = extract_lrs(
    get_warmup_cosine_scheduler(opt, warmup_steps=WARMUP, total_steps=STEPS, min_lr_ratio=0.01),
    STEPS
)

plt.figure(figsize=(10, 4))
plt.plot(warmup_cosine_lrs, color='teal', linewidth=2, label='Warmup+Cosine')
plt.axvline(WARMUP, color='crimson', linestyle='--', linewidth=1.5, label=f'Warmup end (step {WARMUP})')
plt.fill_between(range(WARMUP), 0, warmup_cosine_lrs[:WARMUP], alpha=0.15, color='crimson', label='Warmup region')
plt.title('Linear Warmup + Cosine Annealing  (transformer schedule)')
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.legend(); plt.tight_layout(); plt.show()

---
## 7. ReduceLROnPlateau

Unlike all other schedulers, this is **adaptive** — it monitors a metric (usually validation loss) and reduces the LR when progress stalls.

```
if metric has not improved by `threshold` for `patience` epochs:
    α ← α × factor
```

**Pros**: Automatically handles unknown dynamics  
**Cons**: Requires a validation loop; less predictable

In [ ]:
opt = make_opt(BASE_LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=10, threshold=1e-3
)

# Simulate a validation loss that improves slowly then plateaus
rng = np.random.default_rng(42)
val_losses = (np.exp(-np.linspace(0, 2.5, STEPS))
              + rng.normal(0, 0.015, STEPS)
              + np.where(np.arange(STEPS) > 120, 0.05, 0))  # plateau after step 120

plateau_lrs = extract_lrs(scheduler, STEPS, plateau_losses=val_losses)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(val_losses, color='slategray', linewidth=1.5)
axes[0].set_ylabel('Validation Loss'); axes[0].set_title('Simulated Val Loss')
axes[1].plot(plateau_lrs, color='firebrick', linewidth=2)
axes[1].set_ylabel('Learning Rate'); axes[1].set_xlabel('Step')
axes[1].set_title('ReduceLROnPlateau  (factor=0.5, patience=10)')
plt.tight_layout(); plt.show()

reductions = [i for i in range(1, STEPS) if plateau_lrs[i] < plateau_lrs[i-1]]
print(f'LR reductions at steps: {reductions}')

---
## 8. All Schedulers Side-by-Side

In [ ]:
schedules = {
    'StepLR':                 (step_lrs,          'steelblue'),
    'ExponentialLR':          (exp_lrs,            'darkorange'),
    'CosineAnnealingLR':      (cosine_lrs,         'crimson'),
    'SGDR (T_mult=2)':        (cosine_mult_lrs,    'darkgreen'),
    'OneCycleLR':             (one_cycle_lrs,      'peru'),
    'Warmup+Cosine':          (warmup_cosine_lrs,  'teal'),
    'ReduceLROnPlateau':      (plateau_lrs,        'firebrick'),
}

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (name, (lrs, color)) in zip(axes, schedules.items()):
    ax.plot(lrs, color=color, linewidth=2)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Step'); ax.set_ylabel('LR')

axes[-1].axis('off')
plt.suptitle('Learning Rate Schedulers — Side-by-Side (base_lr=0.1)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 9. Training Demo — Effect of Scheduler on Convergence

Train the same network on the same data with different schedulers and compare loss curves.

In [ ]:
torch.manual_seed(0)

# Simple regression task: learn y = 3x1 + 2x2 + noise
X = torch.randn(512, 8)
w_true = torch.randn(8, 1)
y = X @ w_true + 0.1 * torch.randn(512, 1)

EPOCHS = 200
BATCH  = 64

def train_with_scheduler(scheduler_fn, label, color):
    torch.manual_seed(42)
    net  = nn.Linear(8, 1)
    opt  = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)
    sch  = scheduler_fn(opt)
    criterion = nn.MSELoss()
    losses = []

    dataset = torch.utils.data.TensorDataset(X, y)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True)

    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = criterion(net(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
        sch.step()

    return losses


results = {
    'No Scheduler': train_with_scheduler(
        lambda o: optim.lr_scheduler.LambdaLR(o, lambda _: 1.0), 'No Scheduler', 'gray'),
    'StepLR': train_with_scheduler(
        lambda o: optim.lr_scheduler.StepLR(o, step_size=50, gamma=0.5), 'StepLR', 'steelblue'),
    'CosineAnnealingLR': train_with_scheduler(
        lambda o: optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS, eta_min=1e-4),
        'CosineAnnealingLR', 'crimson'),
    'OneCycleLR': train_with_scheduler(
        lambda o: optim.lr_scheduler.OneCycleLR(o, max_lr=0.1, total_steps=EPOCHS),
        'OneCycleLR', 'darkorange'),
}

plt.figure(figsize=(12, 5))
colors = ['gray', 'steelblue', 'crimson', 'darkorange']
for (name, losses), color in zip(results.items(), colors):
    plt.plot(losses, label=name, color=color, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Train MSE Loss')
plt.title('Scheduler Comparison — Same Model, Same Data')
plt.legend(); plt.yscale('log'); plt.tight_layout(); plt.show()

for name, losses in results.items():
    print(f'{name:20s}  final loss: {losses[-1]:.6f}')

---
## 10. Cosine Annealing — Intuition Visualised

Why does the cosine shape help? The gradient descent trajectory on a loss landscape:

In [ ]:
# Loss landscape for intuition (1D quadratic + perturbation)
theta = np.linspace(-3, 3, 500)
loss_landscape = 0.5 * theta**2 + 0.3 * np.cos(5 * theta)  # bumpy quadratic

# Simulate gradient descent with constant vs cosine LR
def gd_trajectory(lr_schedule, steps=200, theta0=2.5):
    path = [theta0]
    theta_curr = theta0
    for t in range(steps):
        grad = theta_curr - 0.3 * 5 * np.sin(5 * theta_curr)  # d/dtheta
        theta_curr -= lr_schedule[t] * grad
        theta_curr = np.clip(theta_curr, -3, 3)
        path.append(theta_curr)
    return path

TRAJ_STEPS = 200
const_schedule  = [0.02] * TRAJ_STEPS
cosine_schedule = cosine_annealing_manual(0.1, 0.001, TRAJ_STEPS, TRAJ_STEPS).tolist()

path_const  = gd_trajectory(const_schedule)
path_cosine = gd_trajectory(cosine_schedule)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, path, title, color in [
    (axes[0], path_const,  'Constant LR = 0.02',    'steelblue'),
    (axes[1], path_cosine, 'Cosine Annealing LR',   'crimson'),
]:
    ax.plot(theta, loss_landscape, 'k-', linewidth=2, label='Loss landscape')
    ax.plot(path, [0.5*p**2 + 0.3*np.cos(5*p) for p in path],
            'o-', color=color, markersize=2, alpha=0.6, label='GD path')
    ax.plot(path[0], 0.5*path[0]**2 + 0.3*np.cos(5*path[0]),
            's', color='green', markersize=10, label='Start')
    ax.plot(path[-1], 0.5*path[-1]**2 + 0.3*np.cos(5*path[-1]),
            '*', color='gold', markersize=14, label='End')
    ax.set_title(title); ax.set_xlabel('θ'); ax.set_ylabel('Loss')
    ax.legend(fontsize=9)

plt.suptitle('Gradient Descent: Constant LR vs Cosine Annealing', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Constant LR  → final θ = {path_const[-1]:.4f},  loss = {0.5*path_const[-1]**2 + 0.3*np.cos(5*path_const[-1]):.6f}')
print(f'Cosine Anneal → final θ = {path_cosine[-1]:.4f},  loss = {0.5*path_cosine[-1]**2 + 0.3*np.cos(5*path_cosine[-1]):.6f}')

---
## Summary

| Scheduler | When to Use | Key Parameter |
|---|---|---|
| **StepLR** | Simple baselines, small models | `step_size`, `gamma` |
| **ExponentialLR** | When you want smooth, predictable decay | `gamma` (e.g. 0.95–0.99) |
| **CosineAnnealingLR** | Standard choice for modern architectures | `T_max`, `eta_min` |
| **SGDR** | When you want diversity / snapshot ensembles | `T_0`, `T_mult` |
| **OneCycleLR** | Short training runs, finding good LR fast | `max_lr`, `pct_start` |
| **Warmup + Cosine** | Transformers, large batch training | `warmup_steps`, `total_steps` |
| **ReduceLROnPlateau** | Unknown dynamics, transfer learning | `patience`, `factor` |

**Rule of thumb**: For any new project, start with **LinearWarmup + CosineAnnealing**.  
It works well across CNNs, Transformers, and diffusion models alike.